# Дообучение embedding-модели

Этот ноутбук хранит экспериментальный ход подготовки данных, дообучения модели и retrieval-оценки.

Воспроизводимый запуск обучения вынесен в скрипт: `scripts/train_embeddings.py`.

Если нужно повторить обучение без ручного выполнения ячеек, используйте CLI-скрипт. Ноутбук оставлен для анализа промежуточных шагов и проверки решений.

## Подготовка окружения

В этом блоке фиксируются зависимости, которые использовались во время эксперимента. Для воспроизводимого запуска лучше использовать `requirements.txt` и скрипт `scripts/train_embeddings.py`, а не устанавливать пакеты из ячейки notebook.


In [ ]:
# 0. Install
# В отдельной ячейке Jupyter:
#!pip install -U datasets sentence-transformers accelerate pandas scikit-learn

   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 2.1 MB/s  0:00:00
   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 571.3/571.3 kB 4.0 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 4.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.9 MB 3.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.9 MB 3.8 MB/s eta 0:00:03
   ----------- ---------------------------- 2.9/9.9 MB 3.8 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.9 MB 3.8 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.9 MB 3.8 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.9 MB 3.8 MB/s eta 0:00:02
   ------------------------ ----

���⥬� �� 㤠���� ���� 㪠����� ����.


## Импорты

Подключаются базовые библиотеки для работы с путями, табличными данными и Hugging Face datasets. Основные ML-зависимости подключаются ниже в блоках обучения и оценки.


In [4]:
from pathlib import Path
import random
import pandas as pd

from datasets import load_dataset, load_from_disk

## Пути проекта

Определяется корень репозитория и стандартные директории для данных. Пути строятся относительно проекта, чтобы notebook можно было запускать как из корня, так и из папки `notebooks/`.


In [5]:

# 1. Paths
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / ".git").exists() and (path / "README.md").exists():
            return path
    raise RuntimeError("Project root not found")


PROJECT_DIR = find_project_root()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PREPARED_DIR = DATA_DIR / "prepared"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "merionum/ru_paraphraser"
LOCAL_DATASET_PATH = RAW_DIR / "ru_paraphraser_hf"

## Загрузка датасета парафразов

Датасет `merionum/ru_paraphraser` загружается с Hugging Face или читается из локального кэша. Локальное сохранение нужно, чтобы повторные запуски не зависели от сети.


In [6]:
# 2. Download or load local copy

if LOCAL_DATASET_PATH.exists():
    dataset = load_from_disk(str(LOCAL_DATASET_PATH))
    print(f"Loaded from disk: {LOCAL_DATASET_PATH}")
else:
    dataset = load_dataset(DATASET_NAME)
    dataset.save_to_disk(str(LOCAL_DATASET_PATH))
    print(f"Downloaded and saved to: {LOCAL_DATASET_PATH}")

dataset

e:\Mamba\envs\ml-gpu\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\MLCache\huggingface\hub\datasets--merionum--ru_paraphraser. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Saving the dataset (1/1 shards): 100%|██████████| 1924/1924 [00:00<00:00, 480547.90 examples/s]

Downloaded and saved to: E:\ML\Projects\News Flow Analysis\data\raw\ru_paraphraser_hf


DatasetDict({
    train: Dataset({
        features: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class'],
        num_rows: 7227
    })
    test: Dataset({
        features: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class'],
        num_rows: 1924
    })
})

## Первичный просмотр датасета

Проверяются splits, количество строк и имена колонок. Этот шаг нужен перед подготовкой пар, потому что разные версии похожих датасетов могут использовать разные названия текстовых и label-колонок.


In [7]:
# 3. Inspect splits and columns

print(dataset)
for split_name in dataset.keys():
    print("\nSPLIT:", split_name)
    print("Rows:", len(dataset[split_name]))
    print("Columns:", dataset[split_name].column_names)
    print(dataset[split_name][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class'],
        num_rows: 7227
    })
    test: Dataset({
        features: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class'],
        num_rows: 1924
    })
})

SPLIT: train
Rows: 7227
Columns: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class']
{'id': '1', 'id_1': '201', 'id_2': '8159', 'text_1': 'Полицейским разрешат стрелять на поражение по гражданам с травматикой.', 'text_2': 'Полиции могут разрешить стрелять по хулиганам с травматикой.', 'class': '0'}

SPLIT: test
Rows: 1924
Columns: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class']
{'id': '25349', 'id_1': '34420', 'id_2': '34421', 'text_1': 'Цены на нефть восстанавливаются', 'text_2': 'Парламент Словакии поблагодарил народы бывшего СССР за Победу', 'class': '-1'}


## Определение колонок

Функция ищет колонки с первым текстом, вторым текстом и меткой класса. Это делает notebook менее зависимым от конкретных имён колонок в датасете.


In [11]:
def detect_columns(df: pd.DataFrame):
    cols = set(df.columns)

    # Часто у подобных датасетов могут быть text_1/text_2 или sentence1/sentence2.
    possible_text1 = ["text_1", "text1", "sentence1", "sentence_1", "s1"]
    possible_text2 = ["text_2", "text2", "sentence2", "sentence_2", "s2"]
    possible_label = ["label", "class", "target"]

    text1_col = next((c for c in possible_text1 if c in cols), None)
    text2_col = next((c for c in possible_text2 if c in cols), None)
    label_col = next((c for c in possible_label if c in cols), None)

    if text1_col is None or text2_col is None or label_col is None:
        raise ValueError(f"Could not detect columns. Available columns: {df.columns.tolist()}")

    return text1_col, text2_col, label_col


## Подготовка DataFrame

Train split переводится в pandas DataFrame для простой фильтрации, анализа меток и подготовки positive pairs.


In [22]:
# 4. Convert to pandas for quick EDA

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas() if "test" in dataset else None

TEXT1_COL, TEXT2_COL, LABEL_COL = detect_columns(train_df)

train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)

if test_df is not None:
    test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)


display(train_df.head())
print("Text columns:", TEXT1_COL, TEXT2_COL)
print("Label column:", LABEL_COL)
print(train_df[LABEL_COL].dtype)
print(train_df[LABEL_COL].value_counts(dropna=False))

,id,id_1,id_2,text_1,text_2,class
0,1,201,8159,Полицейским разрешат стрелять на поражение по ...,Полиции могут разрешить стрелять по хулиганам ...,0
1,2,202,8158,Право полицейских на проникновение в жилище ре...,Правила внесудебного проникновения полицейских...,0
2,3,273,8167,Президент Египта ввел чрезвычайное положение в...,Власти Египта угрожают ввести в стране чрезвыч...,0
3,4,220,8160,Вернувшихся из Сирии россиян волнует вопрос тр...,Самолеты МЧС вывезут россиян из разрушенной Си...,-1
4,5,223,8160,В Москву из Сирии вернулись 2 самолета МЧС с р...,Самолеты МЧС вывезут россиян из разрушенной Си...,0


Text columns: text_1 text_2
Label column: class
int64
class
 0    2957
-1    2582
 1    1688
Name: count, dtype: int64


## Positive pairs для MultipleNegativesRankingLoss

Для обучения берутся пары с `label == 1`, то есть точные парафразы. В `MultipleNegativesRankingLoss` остальные элементы batch выступают implicit negatives, поэтому явные отрицательные пары напрямую в обучение не передаются.


In [23]:
# 6. Prepare positive pairs for MultipleNegativesRankingLoss

# label == 1: точные парафразы
# label == 0: близкие парафразы, можно добавить позже как weak positives
# label == -1: negative, для MNRL напрямую не нужен

positive_df = train_df[train_df[LABEL_COL] == 1].copy()

positive_df = positive_df[[TEXT1_COL, TEXT2_COL, LABEL_COL]].dropna()
positive_df[TEXT1_COL] = positive_df[TEXT1_COL].astype(str).str.strip()
positive_df[TEXT2_COL] = positive_df[TEXT2_COL].astype(str).str.strip()

positive_df = positive_df[
    (positive_df[TEXT1_COL].str.len() > 0)
    & (positive_df[TEXT2_COL].str.len() > 0)
].reset_index(drop=True)

print("Positive pairs:", len(positive_df))
display(positive_df.head())

Positive pairs: 1688


,text_1,text_2,class
0,Приставы соберут отпечатки пальцев российских ...,Приставы снимут отпечатки пальцев у злостных н...,1
1,Москвичи смогут забронировать в Интернете мест...,В Москве можно будет забронировать место на кл...,1
2,Северокорейский лидер впервые за 19 лет поздра...,Лидер КНДР впервые за 19 лет поздравил согражд...,1
3,Мужчина из Подмосковья случайно убил жену в Но...,Житель Подмосковья случайно убил жену на новог...,1
4,Житель Украины расстрелял посетителей кафе.,На Украине мужчина через окно расстрелял посет...,1


## Сохранение подготовленных пар

Подготовленные positive pairs сохраняются в `data/prepared/`. Эта директория не хранится в Git, потому что содержит воспроизводимые артефакты подготовки данных.


In [25]:
# 7. Save prepared pairs

prepared_pairs_path = PREPARED_DIR / "ru_paraphraser_positive_pairs.parquet"
positive_df.to_parquet(prepared_pairs_path, index=False)

print(prepared_pairs_path)

E:\ML\Projects\News Flow Analysis\data\prepared\ru_paraphraser_positive_pairs.parquet


## Train/eval split

Positive pairs делятся на train и eval. Train используется для дообучения, eval — для retrieval-проверки качества base и fine-tuned моделей.


In [26]:
# 8. Create small train/eval split

from sklearn.model_selection import train_test_split

train_pairs_df, eval_pairs_df = train_test_split(
    positive_df,
    test_size=0.1,
    random_state=42,
    shuffle=True,
)

print(len(train_pairs_df), len(eval_pairs_df))

1519 169


## Первый пробный запуск обучения

Этот блок отражает ранний экспериментальный вариант обучения. Ниже расположен более актуальный вариант с путями относительно проекта и итоговым именем модели.


In [28]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

# базовая модель
BASE_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# куда сохраняем
OUTPUT_DIR = "models/ru-news-mpnet-mnrl"

model = SentenceTransformer(BASE_MODEL_NAME)

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,  # сначала 1
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,  # если упадёт — поставь False
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    eval_strategy="no",
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_pairs,
    loss=loss,
)

trainer.train()

model.save(f"{OUTPUT_DIR}/final")

print("Training finished")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7107.60it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Step,Training Loss
50,0.005267


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]

Training finished


## Дообучение модели

Загружается базовая модель `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` и дообучается на подготовленных парах через `MultipleNegativesRankingLoss`. Финальная модель сохраняется в `models/ru-news-mpnet-paraphrase-mnrl/final`.


In [30]:
# 10. Fine-tuning with Sentence Transformers

from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer

BASE_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
OUTPUT_DIR = PROJECT_DIR / "models" / "ru-news-mpnet-paraphrase-mnrl"

model = SentenceTransformer(BASE_MODEL_NAME)

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,  # если на GPU с поддержкой fp16; если упадёт — поставить False
    bf16=False,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    eval_strategy="no",  # сначала без eval loop, чтобы просто проверить обучение
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_pairs,
    loss=loss,
)

trainer.train()

model.save(str(OUTPUT_DIR / "final"))
print(f"Saved model to: {OUTPUT_DIR / 'final'}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6861.21it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
50,0.005267


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]

Saved model to: E:\ML\Projects\News Flow Analysis\models\ru-news-mpnet-paraphrase-mnrl\final


## Проверка FAISS GPU

Эта ячейка фиксирует попытку установить `faiss-gpu`. В текущем проекте используется `faiss-cpu`, что проще для воспроизводимости на локальной машине и в будущих контейнерах.


In [ ]:
#!pip install faiss-gpu


���⥬� �� 㤠���� ���� 㪠����� ����.
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


## Загрузка моделей для оценки

Для сравнения загружаются базовая модель и дообученная модель. Дальше они оцениваются на одной retrieval-задаче.


In [34]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

BASE_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
FINETUNED_MODEL_PATH = PROJECT_DIR / "models" / "ru-news-mpnet-paraphrase-mnrl" / "final"

base_model = SentenceTransformer(BASE_MODEL_NAME)
ft_model = SentenceTransformer(FINETUNED_MODEL_PATH)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6633.13it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6862.12it/s]


## Формирование retrieval-задачи

Для каждого query известен правильный positive. Кандидаты состоят из правильных positives и дополнительных distractors из отрицательных примеров.


In [35]:
# eval_pairs_df должен быть из твоего train_test_split
queries = eval_pairs_df[TEXT1_COL].astype(str).tolist()
positives = eval_pairs_df[TEXT2_COL].astype(str).tolist()

# кандидаты: все правильные positives + немного distractors из negative class
negatives_pool = train_df[train_df[LABEL_COL] == -1][TEXT2_COL].astype(str).dropna().tolist()

import random
random.seed(42)

extra_negatives = random.sample(
    negatives_pool,
    min(2000, len(negatives_pool))
)

candidates = positives + extra_negatives

# Уберём дубли, но сохраним порядок
seen = set()
unique_candidates = []
for x in candidates:
    if x not in seen:
        unique_candidates.append(x)
        seen.add(x)

candidates = unique_candidates

# Для каждого query запомним индекс правильного positive
candidate_index = {text: i for i, text in enumerate(candidates)}
target_indices = [candidate_index[p] for p in positives]

print("Queries:", len(queries))
print("Candidates:", len(candidates))

Queries: 169
Candidates: 1768


## Retrieval-оценка

Embeddings нормализуются, после чего FAISS `IndexFlatIP` используется как cosine similarity search. Метрики: Recall@1, Recall@5, Recall@10 и MRR@10.


In [36]:
def evaluate_retrieval(model, queries, candidates, target_indices, top_k=10, batch_size=64):
    query_emb = model.encode(
        queries,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    candidate_emb = model.encode(
        candidates,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    dim = candidate_emb.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine для normalized vectors
    index.add(candidate_emb)

    scores, indices = index.search(query_emb, top_k)

    recall_at_1 = 0
    recall_at_5 = 0
    recall_at_10 = 0
    mrr = 0.0

    for i, retrieved in enumerate(indices):
        target = target_indices[i]

        if target == retrieved[0]:
            recall_at_1 += 1

        if target in retrieved[:5]:
            recall_at_5 += 1

        if target in retrieved[:10]:
            recall_at_10 += 1

        rank_positions = np.where(retrieved == target)[0]
        if len(rank_positions) > 0:
            rank = rank_positions[0] + 1
            mrr += 1.0 / rank

    n = len(queries)

    return {
        "Recall@1": recall_at_1 / n,
        "Recall@5": recall_at_5 / n,
        "Recall@10": recall_at_10 / n,
        "MRR@10": mrr / n,
    }

## Сравнение base и fine-tuned моделей

Обе модели оцениваются на одинаковом наборе queries и candidates. Это позволяет проверить, дало ли дообучение прирост качества на задаче поиска парафразов.


In [37]:
base_metrics = evaluate_retrieval(
    base_model,
    queries,
    candidates,
    target_indices,
    top_k=10,
)

ft_metrics = evaluate_retrieval(
    ft_model,
    queries,
    candidates,
    target_indices,
    top_k=10,
)

print("Base:", base_metrics)
print("Fine-tuned:", ft_metrics)

Batches: 100%|██████████| 28/28 [00:00<00:00, 37.37it/s]

Base: {'Recall@1': 0.9526627218934911, 'Recall@5': 0.9940828402366864, 'Recall@10': 0.9940828402366864, 'MRR@10': np.float64(0.9699211045364893)}
Fine-tuned: {'Recall@1': 0.9644970414201184, 'Recall@5': 1.0, 'Recall@10': 1.0, 'MRR@10': np.float64(0.979783037475345)}


## Таблица метрик

Итоговые метрики выводятся в компактном виде для дальнейшего переноса в документацию проекта.


In [38]:
pd.DataFrame(
    [base_metrics, ft_metrics],
    index=["base", "fine_tuned"]
)


,Recall@1,Recall@5,Recall@10,MRR@10
base,0.952663,0.994083,0.994083,0.969921
fine_tuned,0.964497,1.000000,1.000000,0.979783
